# Project: Identify Customer Segments

In this project, you will apply unsupervised learning techniques to identify segments of the population that form the core customer base for a mail-order sales company in Germany. These segments can then be used to direct marketing campaigns towards audiences that will have the highest expected rate of returns. The data that you will use has been provided by our partners at Bertelsmann Arvato Analytics, and represents a real-life data science task.

This notebook will help you complete this task by providing a framework within which you will perform your analysis steps. In each step of the project, you will see some text describing the subtask that you will perform, followed by one or more code cells for you to complete your work. **Feel free to add additional code and markdown cells as you go along so that you can explore everything in precise chunks.** The code cells provided in the base template will outline only the major tasks, and will usually not be enough to cover all of the minor tasks that comprise it.

It should be noted that while there will be precise guidelines on how you should handle certain tasks in the project, there will also be places where an exact specification is not provided. **There will be times in the project where you will need to make and justify your own decisions on how to treat the data.** These are places where there may not be only one way to handle the data. In real-life tasks, there may be many valid ways to approach an analysis task. One of the most important things you can do is clearly document your approach so that other scientists can understand the decisions you've made.

At the end of most sections, there will be a Markdown cell labeled **Discussion**. In these cells, you will report your findings for the completed section, as well as document the decisions that you made in your approach to each subtask. **Your project will be evaluated not just on the code used to complete the tasks outlined, but also your communication about your observations and conclusions at each stage.**

In [ ]:
# import libraries here; add more as necessary
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
# from tqdm import tqdm

from tqdm.auto import tqdm
from pathlib import Path # for file paths
import shutil # for moving files
import ast # for parsing the missing or unknown data
import re # for regular expressions
from sklearn.cluster import KMeans
import gc # for garbage collection when we delete the old good_pca data


# magic word for producing visualizations in notebook
%matplotlib inline

pd.options.display.max_rows = None # show all rows
pd.options.display.max_columns = None # show all columns

In [ ]:
# check if the data is in the ./data subfolder. If not create the ./data folder and move the 4 files over for a cleaner file structure:

data_dir = Path("data")
data_dir.mkdir(exist_ok=True) # create the data folder if it doesn't exist

expected_files = [
    "Udacity_AZDIAS_Subset.csv",
    "Udacity_CUSTOMERS_Subset.csv",
    "AZDIAS_Feature_Summary.csv",
    "Data_Dictionary.md",
]

for filename in expected_files:
    dest = data_dir / filename
    src = Path(filename)

    if dest.exists():
        print(f"Found: {dest}")
    elif src.exists():
        shutil.move(str(src), str(dest))
        print(f"Moved: {src} -> {dest}")
    else:
        print(f"Couldn't find {filename}. This notebook will not work if the csv files are not in the ./data folder.")


### Step 0: Load the Data

There are four files associated with this project (not including this one):

- `Udacity_AZDIAS_Subset.csv`: Demographics data for the general population of Germany; 891211 persons (rows) x 85 features (columns).
- `Udacity_CUSTOMERS_Subset.csv`: Demographics data for customers of a mail-order company; 191652 persons (rows) x 85 features (columns).
- `Data_Dictionary.md`: Detailed information file about the features in the provided datasets.
- `AZDIAS_Feature_Summary.csv`: Summary of feature attributes for demographics data; 85 features (rows) x 4 columns

Each row of the demographics files represents a single person, but also includes information outside of individuals, including information about their household, building, and neighborhood. You will use this information to cluster the general population into groups with similar demographic properties. Then, you will see how the people in the customers dataset fit into those created clusters. The hope here is that certain clusters are over-represented in the customers data, as compared to the general population; those over-represented clusters will be assumed to be part of the core userbase. This information can then be used for further applications, such as targeting for a marketing campaign.

To start off with, load in the demographics data for the general population into a pandas DataFrame, and do the same for the feature attributes summary. Note for all of the `.csv` data files in this project: they're semicolon (`;`) delimited, so you'll need an additional argument in your [`read_csv()`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.read_csv.html) call to read in the data properly. Also, considering the size of the main dataset, it may take some time for it to load completely.

Once the dataset is loaded, it's recommended that you take a little bit of time just browsing the general structure of the dataset and feature summary file. You'll be getting deep into the innards of the cleaning in the first major step of the project, so gaining some general familiarity can help you get your bearings.

In [ ]:
# Load in the general demographics data.
gen_demo = pd.read_csv("data/Udacity_AZDIAS_Subset.csv", sep=";")

# Load in the feature summary file and store it to a variable
# named `feat_info`.
feat_info = pd.read_csv("data/AZDIAS_Feature_Summary.csv", sep=";")



In [ ]:
# Check the structure of the data after it's loaded (e.g. print the number of
# rows and columns, print the first few rows).
gen_demo.head()


In [ ]:
gen_demo.shape


In [ ]:
gen_demo.info()

In [ ]:
gen_demo.describe()

In [ ]:
feat_info

In [ ]:
feat_info.shape

In [ ]:
feat_info['missing_or_unknown'].dtype


> **Tip**: Add additional cells to keep everything in reasonably-sized chunks! Keyboard shortcut `esc --> a` (press escape to enter command mode, then press the 'A' key) adds a new cell before the active cell, and `esc --> b` adds a new cell after the active cell. If you need to convert an active cell to a markdown cell, use `esc --> m` and to convert to a code cell, use `esc --> y`. 

## Step 1: Preprocessing

### Step 1.1: Assess Missing Data

The feature summary file contains a summary of properties for each demographics data column. You will use this file to help you make cleaning decisions during this stage of the project. First of all, you should assess the demographics data in terms of missing data. Pay attention to the following points as you perform your analysis, and take notes on what you observe. Make sure that you fill in the **Discussion** cell with your findings and decisions at the end of each step that has one!

#### Step 1.1.1: Convert Missing Value Codes to NaNs
The fourth column of the feature attributes summary (loaded in above as `feat_info`) documents the codes from the data dictionary that indicate missing or unknown data. While the file encodes this as a list (e.g. `[-1,0]`), this will get read in as a string object. You'll need to do a little bit of parsing to make use of it to identify and clean the data. Convert data that matches a 'missing' or 'unknown' value code into a numpy NaN value. You might want to see how much data takes on a 'missing' or 'unknown' code, and how much data is naturally missing, as a point of interest.

**As one more reminder, you are encouraged to add additional cells to break up your analysis into manageable chunks.**

In [ ]:
# Identify missing or unknown data values and convert them to NaNs.

feat_info_clean = feat_info.copy()
feat_info_clean['missing_or_unknown'] = feat_info_clean['missing_or_unknown'].astype(object)

for i in range(len(feat_info)):
    missing_or_unknown = feat_info.iloc[i]['missing_or_unknown']

    # CAMEO columns use letter codes (X, XX) — quote them so literal_eval works
    if re.search(r'[A-Za-z]', missing_or_unknown):
        missing_or_unknown = re.sub(
            r'(?<=[\[,])\s*([A-Za-z]+)\s*(?=[,\]])',
            r"'\1'",
            missing_or_unknown,
        )

    missing_or_unknown = ast.literal_eval(missing_or_unknown)
    feat_info_clean.at[i, 'missing_or_unknown'] = missing_or_unknown

feat_info_clean


In [ ]:
# double check that it is a list and no longer a string

print("The type is", type(feat_info_clean['missing_or_unknown'][59]))
if type(feat_info_clean['missing_or_unknown'][59]) == list:
    print("Woohoo!")
else:
    print("Doh!")


In [ ]:
# now let's work on the data
gen_demo_clean = gen_demo.copy()
nan_count = 0
cols_with_missing_values = dict()

for feature, missing_codes in tqdm(
    zip(feat_info_clean['attribute'], feat_info_clean['missing_or_unknown']), # loop through the features and the missing codes
    total=len(feat_info_clean), # total number of features
    desc="Cleaning columns", # description of the loop
):

    if missing_codes:
        matches = gen_demo_clean[feature].isin(missing_codes).sum() # this counts the number of missing codes in the feature
        gen_demo_clean[feature] = gen_demo_clean[feature].replace(missing_codes, np.nan) # this replaces the missing codes with NaN in all the rows of the feature
        nan_count += matches # update the total number of NaN values
        cols_with_missing_values[feature] = matches # update the dictionary with the number of missing values for each feature

print(f"Number of NaN values: {nan_count}")

# convert to df for sorting ...
cols_with_missing_df = (
    pd.DataFrame.from_dict(cols_with_missing_values, orient="index", columns=["missing_count"])
    .sort_values("missing_count", ascending=False)
)

print(f"Columns with missing values: {cols_with_missing_df}")

gen_demo_clean.head()

#### Step 1.1.2: Assess Missing Data in Each Column

How much missing data is present in each column? There are a few columns that are outliers in terms of the proportion of values that are missing. You will want to use matplotlib's [`hist()`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.hist.html) function to visualize the distribution of missing value counts to find these columns. Identify and document these columns. While some of these columns might have justifications for keeping or re-encoding the data, for this project you should just remove them from the dataframe. (Feel free to make remarks about these outlier columns in the discussion, however!)

For the remaining features, are there any patterns in which columns have, or share, missing data?

In [ ]:
# Perform an assessment of how much missing data there is in each column of the
# dataset.

missing_per_column = gen_demo_clean.isnull().sum().sort_values(ascending=False)
missing_pct_per_column = (missing_per_column / len(gen_demo_clean)).sort_values(ascending=False)

# let's create a summary of the missing data with the missing count and the missing percentage
missing_summary = pd.DataFrame({
    "missing_count": missing_per_column,
    "missing_pct": missing_pct_per_column.round(2)*100
}).sort_values(by="missing_count", ascending=False)

print(missing_summary)

# Visualize missing data per column
plt.figure(figsize=(18, 8))
plt.bar(missing_per_column.index, missing_per_column.to_numpy(), color="orange")
plt.xlabel("Column")
plt.ylabel("Number of missing values")
plt.title("Missing data per column")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
# Investigate patterns in the amount of missing data in each column.

missing_mask = gen_demo_clean.isnull()
# Keep only columns that have at least one missing value
cols_with_missing = (
    missing_mask.sum()
    .sort_values(ascending=False)
    .loc[lambda s: s > 0]
    .index
)
# now calculate the correlation of the missing values. .corr() calculates the correlation of the missing values.
missing_corr = missing_mask.loc[:, cols_with_missing].corr()
# Sort by missing count (highest first)
missing_corr_sorted = missing_corr.loc[cols_with_missing, cols_with_missing]
plt.figure(figsize=(14, 14))
sns.heatmap(
    missing_corr_sorted,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
)
plt.title("Correlation of missing values")
plt.xticks(rotation=90) # rotate the x-axis labels 90 degrees so they are vertical
plt.tight_layout()
plt.show()

In [ ]:
# Pairs of columns that are missing together (excluding self-correlation)
corr_pairs = (
    missing_corr.where(np.triu(np.ones(missing_corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
print("Top column pairs with shared missingness:")
print(corr_pairs.head(120))

In [ ]:
# Remove the outlier columns from the dataset. (You'll perform other data
# engineering tasks such as re-encoding and imputation later.)

missing_threshold = 0.20 # looking at the histogram we can see that the columns with more than 20% missing values are outliers
outlier_columns = missing_pct_per_column[missing_pct_per_column > missing_threshold].index.tolist()

print(f"\nColumns we are removing >{missing_threshold:.0%}:")
print(outlier_columns)

gen_demo_clean = gen_demo_clean.drop(columns=outlier_columns)
print(f"\nShape after dropping outlier columns: {gen_demo_clean.shape}")



#### Discussion 1.1.2: Assess Missing Data in Each Column

Most columns had little missing data. However, there were some columns that had a large amount of data missing and therefore were removed. I chose >0.20 as the threshold to remove them:

- TITEL_KZ (Academic title flag) almost all data missing - so pretty useless
- AGER_TYP (Best-ager typology) with ~77% missing
- KK_KUNDENTYP (Consumer pattern over past 12 months) with ~66% missing
- KBA05_BAUMAX (Most common building type within the microcell) ~53% missing
- GEBURTSJAHR (Year of birth) with ~44% missing
- ALTER_HH (Birthdate of head of household) with ~35% missing

Regarding Geburtsjahr (birth year) and ALTER_HH (head of household age): According to the heatmap there is not too much correlation between the two in regard to missing data so we theoretically could create a new feature "estimated age" and combine the two estimating the age for the rows with missing age data based on the birth year. 

When taking a closer look at the correlation heatmap we can see some patterns:

- KBA05_ANTG1 to ANTG4, KBA05_GBZ, and MOBI_REGIO seem to be correlated for missing data. Intuition is that housing data as a whole was missing (not collected) for those data records
- KKK and REGIOTYP have exactly the same missing data count (158064) indicating a shared data gap
- GEBAEUDETYP, MIN_GEBAEUDEJAHR, WOHNLAGE, OST_WEST_KZ also seem to be correlated for missing data
- The PLZ8_* features seem to be either present as a block of features or not at all  




#### Step 1.1.3: Assess Missing Data in Each Row

Now, you'll perform a similar assessment for the rows of the dataset. How much data is missing in each row? As with the columns, you should see some groups of points that have a very different numbers of missing values. Divide the data into two subsets: one for data points that are above some threshold for missing values, and a second subset for points below that threshold.

In order to know what to do with the outlier rows, we should see if the distribution of data values on columns that are not missing data (or are missing very little data) are similar or different between the two groups. Select at least five of these columns and compare the distribution of values.
- You can use seaborn's [`countplot()`](https://seaborn.pydata.org/generated/seaborn.countplot.html) function to create a bar chart of code frequencies and matplotlib's [`subplot()`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.subplot.html) function to put bar charts for the two subplots side by side.
- To reduce repeated code, you might want to write a function that can perform this comparison, taking as one of its arguments a column to be compared.

Depending on what you observe in your comparison, this will have implications on how you approach your conclusions later in the analysis. If the distributions of non-missing features look similar between the data with many missing values and the data with few or no missing values, then we could argue that simply dropping those points from the analysis won't present a major issue. On the other hand, if the data with many missing values looks very different from the data with few or no missing values, then we should make a note on those data as special. We'll revisit these data later on. **Either way, you should continue your analysis for now using just the subset of the data with few or no missing values.**

In [ ]:
# How much data is missing in each row of the dataset?

# check the missing values in each ROW
missing_per_row = gen_demo_clean.isnull().sum(axis=1) #.sort_values(ascending=False)
# print(missing_per_row.describe)

# count the number of rows with each number of missing values
missing_per_row.value_counts()

In [ ]:
# plot the distribution of the missing values in each row
plt.figure(figsize=(10, 6))
sns.histplot(x=missing_per_row, color="orange", edgecolor="none", bins=int(missing_per_row.max()))
plt.xlabel("Number of missing values per row")
plt.ylabel("Number of rows")
plt.title("Distribution of missing values per row")
plt.show()


In [ ]:
# summarize the distribution of the missing values in each row
missing_per_row.describe()

In [ ]:
# Write code to divide the data into two subsets based on the number of missing
# values in each row.

# first let's create the good and bad rows
good_rows = missing_per_row.loc[missing_per_row <= 25]
bad_rows = missing_per_row.loc[missing_per_row > 25]

# compare the distribution of missing values in each subset (seaborn, same style as below)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

sns.histplot(x=good_rows, ax=axes[0], color="orange", edgecolor="none", bins=int(good_rows.max()))
axes[0].set_title("Good rows (≤ 25 missing)")
axes[0].set_xlabel("Number of missing values per row")
axes[0].set_ylabel("Number of rows")

sns.histplot(x=bad_rows, ax=axes[1], color="gray", edgecolor="none", bins=int(bad_rows.max()))
axes[1].set_title("Bad rows (> 25 missing)")
axes[1].set_xlabel("Number of missing values per row")
axes[1].set_ylabel("Number of rows")

plt.tight_layout()
plt.show()

print(f"Number of good rows: {good_rows.shape[0]}")
print(f"Number of bad rows: {bad_rows.shape[0]}")



In [ ]:
good_rows_df = gen_demo_clean.loc[good_rows.index]
bad_rows_df = gen_demo_clean.loc[bad_rows.index]

print(f"Good rows shape: {good_rows_df.shape}")
print(f"Bad rows shape: {bad_rows_df.shape}")

good_rows_df.head()

In [ ]:
# now lets get the full data frames with the indexes of the good and bad rows

good_df = gen_demo_clean.loc[good_rows.index]
bad_df = gen_demo_clean.loc[bad_rows.index]

print(f"Good df shape: {good_df.shape}")
print(f"Bad df shape: {bad_df.shape}")
print(good_df.shape)
print(bad_df.shape)

In [ ]:
# Compare the distribution of values for at least five columns where there are
# no or few missing values, between the two subsets.

few_missing_cols = missing_per_column.loc[missing_per_column <= 25].index[:10].tolist()

print(f"We are comparing the following columns: {few_missing_cols}")

# loop over the first 10 columns with few missing values and plot the distribution of the values
for col in few_missing_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

    # plot the good rows
    sns.countplot(data=good_df, x=col, stat="percent", ax=axes[0], color="orange")
    axes[0].set_title(f"Good rows: {col}")
    axes[0].set_ylabel("%")

    # plot the bad rows
    sns.countplot(data=bad_df, x=col, stat="percent", ax=axes[1], color="gray")
    axes[1].set_title(f"Bad rows: {col}")
    axes[1].set_ylabel("%")

    plt.tight_layout()
    plt.show()



#### Discussion 1.1.3: Assess Missing Data in Each Row

Most rows have very little missing data. Using 25 missing values per row as the threshold, ~90% of records (797,961 rows) have <= 25 missing values ("good rows"), while ~10% (93,260 rows) have > 25 missing values ("bad rows").

I compared the distributions (as percentages) of ten mostly-complete columns between the two subsets: FINANZ_UNAUFFAELLIGER, FINANZTYP, FINANZ_HAUSBAUER, GREEN_AVANTGARDE, FINANZ_SPARER, FINANZ_MINIMALIST, FINANZ_VORSORGER, FINANZ_ANLEGER, ANREDE_KZ, and SEMIO_KAEM.

There is a clear difference: With the exception of ANREDE_KZ (salutation which is basically gender), the bad rows are not well distributed and one value dominates (~80%) in most columns. Good rows show more balanced distributions across categories.

High-missing rows appear qualitatively different, not just incomplete versions of the same data. One possibility could be that those are default values that were used when the data was not collected/was missing. I will continue the analysis using the "good rows" subset and keep the high-missing group only as an extra segment if useful later on.

### Step 1.2: Select and Re-Encode Features

Checking for missing data isn't the only way in which you can prepare a dataset for analysis. Since the unsupervised learning techniques to be used will only work on data that is encoded numerically, you need to make a few encoding changes or additional assumptions to be able to make progress. In addition, while almost all of the values in the dataset are encoded using numbers, not all of them represent numeric values. Check the third column of the feature summary (`feat_info`) for a summary of types of measurement.
- For numeric and interval data, these features can be kept without changes.
- Most of the variables in the dataset are ordinal in nature. While ordinal values may technically be non-linear in spacing, make the simplifying assumption that the ordinal variables can be treated as being interval in nature (that is, kept without any changes).
- Special handling may be necessary for the remaining two variable types: categorical, and 'mixed'.

In the first two parts of this sub-step, you will perform an investigation of the categorical and mixed-type features and make a decision on each of them, whether you will keep, drop, or re-encode each. Then, in the last part, you will create a new data frame with only the selected and engineered columns.

Data wrangling is often the trickiest part of the data analysis process, and there's a lot of it to be done here. But stick with it: once you're done with this step, you'll be ready to get to the machine learning parts of the project!

In [ ]:
# check what type of data each feature is
feat_info

In [ ]:
# How many features are there of each data type?

print("Data type counts:")
feat_info['type'].value_counts().sort_index()



#### Step 1.2.1: Re-Encode Categorical Features

For categorical data, you would ordinarily need to encode the levels as dummy variables. Depending on the number of categories, perform one of the following:
- For binary (two-level) categoricals that take numeric values, you can keep them without needing to do anything.
- There is one binary variable that takes on non-numeric values. For this one, you need to re-encode the values as numbers or create a dummy variable.
- For multi-level categoricals (three or more values), you can choose to encode the values using multiple dummy variables (e.g. via [OneHotEncoder](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)), or (to keep things straightforward) just drop them from the analysis. As always, document your choices in the Discussion section.

In [ ]:
# re-read the good_df so we can run this cell multiple times as we drop columns in the next cell
good_df = gen_demo_clean.loc[good_rows.index] # this is the good_df from Step 1.1.3

# create a list of categorical features (column names)
categorical_features = feat_info[feat_info['type'] == 'categorical']['attribute'].tolist()

# we dropped some columns in Step 1.1, so we need to update the list of categorical features
categorical_features = [col for col in categorical_features if col in good_df.columns]

categorical_features

In [ ]:
# Assess categorical variables: which are binary, which are multi-level, and
# which one needs to be re-encoded?

# Always re-run the cell above to get the latest good_df before running this cell

cat_summary = []
for col in categorical_features:
    # drop the missing values
    non_null = good_df[col].dropna()

    # get the unique values
    unique_values = non_null.unique()

    # get the number of levels
    num_levels = len(unique_values)
    is_numeric = pd.api.types.is_numeric_dtype(good_df[col])

    # check if the feature is binary
    if num_levels == 2:
        category = "binary_numeric (keep)" if is_numeric else "binary_non_numeric (re-encode)"
        # print(f"{col} is binary and {is_numeric=}")
    else:
        category = "multi_level (dummy variables or drop)"
        # print(f"{col} is multi-level and {is_numeric=}")
    # append the feature to the summary
    cat_summary.append({
        "column": col,
        "num_levels": num_levels,
        "dtype": good_df[col].dtype,
        "category": category,
        "unique_values": sorted(unique_values, key=str),
    })

# create a dataframe from the summary
cat_summary_df = pd.DataFrame(cat_summary)

# get the binary features
binary_features = cat_summary_df.loc[
    cat_summary_df["category"].str.startswith("binary"), "column"
].tolist()

# get the multi-level features
multi_level_features = cat_summary_df.loc[
    cat_summary_df["category"] == "multi_level (dummy variables or drop)", "column"
].tolist()

# get the features that need to be re-encoded (binary non-numeric)
needs_reencode = cat_summary_df.loc[
    cat_summary_df["category"] == "binary_non_numeric (re-encode)", "column"
].tolist()

print("Binary:", binary_features)
print("Multi-level:", multi_level_features)

for level in multi_level_features:
    print(f"  - {level} has {good_df[level].nunique()} levels")

print("Shall be re-encoded:", needs_reencode)


In [ ]:
# Re-encode categorical variable(s) to be kept in the analysis.

# Work on the good-rows subset going forward
df = good_df.copy()

# Re-encode the non-numeric binary: OST_WEST_KZ (O=East, W=West)
df["OST_WEST_KZ"] = df["OST_WEST_KZ"].map({"O": 0, "W": 1})

# Drop multi-level categoricals (simplest approach)
df = df.drop(columns=multi_level_features)

# Update good_df for later steps
good_df = df

print(f"Shape after re-encoding/dropping: {good_df.shape}")
print("OST_WEST_KZ unique values:", good_df["OST_WEST_KZ"].dropna().unique())
print("ANREDE_KZ unique values:", good_df["ANREDE_KZ"].dropna().unique()) # will be re-encoded in Step 2 with the StandardScaler to [0, 1]


#### Discussion 1.2.1: Re-Encode Categorical Features

I explored the categorical features and split them into binary and multi-level. 

The binary numeric columns were kept. OST_WEST_KZ was re-encoded from ['O', 'W'] to [0, 1]. ANREDE_KZ was kept as it is already numeric binary.

The multi-level categoricals (GEBAEUDETYP, LP_*, CAMEO_DEU_2015, etc.) were dropped to avoid a large dummy-variable/one-hot expansion.

CAMEO_INTL_2015 will be handled separately as a mixed feature in Step 1.2.2.

#### Step 1.2.2: Engineer Mixed-Type Features

There are a handful of features that are marked as "mixed" in the feature summary that require special treatment in order to be included in the analysis. There are two in particular that deserve attention; the handling of the rest are up to your own choices:
- "PRAEGENDE_JUGENDJAHRE" combines information on three dimensions: generation by decade, movement (mainstream vs. avantgarde), and nation (east vs. west). While there aren't enough levels to disentangle east from west, you should create two new variables to capture the other two dimensions: an interval-type variable for decade, and a binary variable for movement.
- "CAMEO_INTL_2015" combines information on two axes: wealth and life stage. Break up the two-digit codes by their 'tens'-place and 'ones'-place digits into two new ordinal variables (which, for the purposes of this project, is equivalent to just treating them as their raw numeric values).
- If you decide to keep or engineer new features around the other mixed-type features, make sure you note your steps in the Discussion section.

Be sure to check `Data_Dictionary.md` for the details needed to finish these tasks.

In [ ]:
good_df["PRAEGENDE_JUGENDJAHRE"].value_counts()

In [ ]:
good_df["PRAEGENDE_JUGENDJAHRE"].head()

In [ ]:
# good_df["PRAEGENDE_JUGENDJAHRE"]

```PRAEGENDE_JUGENDJAHRE```

Dominating movement of person's youth (avantgarde vs. mainstream; east vs. west)

- -1: unknown
- 0: unknown
- 1: 40s - war years (Mainstream, E+W)
- 2: 40s - reconstruction years (Avantgarde, E+W)
- 3: 50s - economic miracle (Mainstream, E+W)
- 4: 50s - milk bar / Individualisation (Avantgarde, E+W)
- 5: 60s - economic miracle (Mainstream, E+W)
- 6: 60s - generation 68 / student protestors (Avantgarde, W)
- 7: 60s - opponents to the building of the Wall (Avantgarde, E)
- 8: 70s - family orientation (Mainstream, E+W)
- 9: 70s - peace movement (Avantgarde, E+W)
- 10: 80s - Generation Golf (Mainstream, W)
- 11: 80s - ecological awareness (Avantgarde, W)
- 12: 80s - FDJ / communist party youth organisation (Mainstream, E)
- 13: 80s - Swords into ploughshares (Avantgarde, E)
- 14: 90s - digital media kids (Mainstream, E+W)
- 15: 90s - ecological awareness (Avantgarde, E+W)

In [ ]:
# Investigate PRAEGENDE_JUGENDJAHRE (see above) and engineer two new variables.

col = "PRAEGENDE_JUGENDJAHRE"

# Map combined codes to decade (40s-90s) and movement (0=mainstream, 1=avantgarde)
decade_map = {
    1: 40, 2: 40,
    3: 50, 4: 50,
    5: 60, 6: 60, 7: 60,
    8: 70, 9: 70,
    10: 80, 11: 80, 12: 80, 13: 80,
    14: 90, 15: 90,
}
mainstream = {1, 3, 5, 8, 10, 12, 14}
avantgarde = {2, 4, 6, 7, 9, 11, 13, 15}

# let's create two new feature PRAEGENDE_JUGENDJAHRE_DECADE 
good_df[f"{col}_DECADE"] = good_df[col].map(decade_map)


# let's create two new feature PRAEGENDE_JUGENDJAHRE_MOVEMENT
def youth_movement(code):
    if code in avantgarde:
        return 1
    if code in mainstream:
        return 0
    return np.nan

good_df[f"{col}_MOVEMENT"] = good_df[col].map(youth_movement)

# print the first 5 rows of the new features
print(good_df[[col, f"{col}_DECADE", f"{col}_MOVEMENT"]].dropna().head())


In [ ]:
good_df["CAMEO_INTL_2015"].value_counts()

In [ ]:
good_df["CAMEO_INTL_2015"].head()


```CAMEO_INTL_2015```

German CAMEO: Wealth / Life Stage Typology, mapped to international code

- -1: unknown
- 11: Wealthy Households - Pre-Family Couples & Singles
- 12: Wealthy Households - Young Couples With Children
- 13: Wealthy Households - Families With School Age Children
- 14: Wealthy Households - Older Families &  Mature Couples
- 15: Wealthy Households - Elders In Retirement
- 21: Prosperous Households - Pre-Family Couples & Singles
- 22: Prosperous Households - Young Couples With Children
- 23: Prosperous Households - Families With School Age Children
- 24: Prosperous Households - Older Families & Mature Couples
- 25: Prosperous Households - Elders In Retirement
- 31: Comfortable Households - Pre-Family Couples & Singles
- 32: Comfortable Households - Young Couples With Children
- 33: Comfortable Households - Families With School Age Children
- 34: Comfortable Households - Older Families & Mature Couples
- 35: Comfortable Households - Elders In Retirement
- 41: Less Affluent Households - Pre-Family Couples & Singles
- 42: Less Affluent Households - Young Couples With Children
- 43: Less Affluent Households - Families With School Age Children
- 44: Less Affluent Households - Older Families & Mature Couples
- 45: Less Affluent Households - Elders In Retirement
- 51: Poorer Households - Pre-Family Couples & Singles
- 52: Poorer Households - Young Couples With Children
- 53: Poorer Households - Families With School Age Children
- 54: Poorer Households - Older Families & Mature Couples
- 55: Poorer Households - Elders In Retirement
- XX: unknown

In [ ]:
# Investigate "CAMEO_INTL_2015" and engineer two new variables.

col = "CAMEO_INTL_2015"

# let's create a new feature CAMEO_INTL_2015_WEALTH 
wealth_map = {
  "11": 1, "12": 1, "13": 1, "14": 1, "15": 1,
  "21": 2, "22": 2, "23": 2, "24": 2, "25": 2,
  "31": 3, "32": 3, "33": 3, "34": 3, "35": 3,
  "41": 4, "42": 4, "43": 4, "44": 4, "45": 4,
  "51": 5, "52": 5, "53": 5, "54": 5, "55": 5,
  "XX": np.nan
}

good_df[f"{col}_WEALTH"] = good_df[col].map(wealth_map)

# let's create a new feature CAMEO_INTL_2015_LIFE_STAGE
life_stage_map = {
  "11": 1, "12": 2, "13": 3, "14": 4, "15": 5,
  "21": 1, "22": 2, "23": 3, "24": 4, "25": 5,
  "31": 1, "32": 2, "33": 3, "34": 4, "35": 5,
  "41": 1, "42": 2, "43": 3, "44": 4, "45": 5,
  "51": 1, "52": 2, "53": 3, "54": 4, "55": 5,
  "XX": np.nan
}

good_df[f"{col}_LIFE_STAGE"] = good_df[col].map(life_stage_map)

# print the first 5 rows of the new features
print(good_df[[col, f"{col}_WEALTH", f"{col}_LIFE_STAGE"]].dropna().head())


#### Discussion 1.2.2: Engineer Mixed-Type Features

#### PRAEGENDE_JUGENDJAHRE

Each code 1-15 combines decade, movement, and region (east/west). With the last one: east vs west could not be cleanly separated from these codes. So, I only created new variables for PRAEGENDE_JUGENDJAHRE_DECADE (40, 50, 60, 70, 80, or 90) and PRAEGENDE_JUGENDJAHRE_MOVEMENT (0 = mainstream, 1 = avantgarde).


#### CAMEO_INTL_2015

Each code is a two-digit code with tens digit representing the wealth: 1 (wealthy) to 5 (poorer) and the ones digit the life stage 1 (pre-family) to 5 (elders). Here I created two variables CAMEO_INTL_2015_WEALTH (1 - 5) and CAMEO_INTL_2015_LIFE_STAGE (1 - 5). XX was mapped to NaN.


#### Other mixed types

Other mixed types were kept without engineering new features. 

- LP_LEBENSPHASE_FEIN and LP_LEBENSPHASE_GROB: already numeric life-phase codes
- WOHNLAGE: Neighborhood quality already has ordinal values 1 to 5
- PLZ8_BAUMAX: Building-type code already has ordinal values

And we had already dropped KBA05_BAUMAX earlier due to >20% missing values.


#### Original columns to drop

The original PRAEGENDE_JUGENDJAHRE and CAMEO_INTL_2015 columns will be removed in Step 1.2.3 below.


#### Step 1.2.3: Complete Feature Selection

In order to finish this step up, you need to make sure that your data frame now only has the columns that you want to keep. To summarize, the dataframe should consist of the following:
- All numeric, interval, and ordinal type columns from the original dataset.
- Binary categorical features (all numerically-encoded).
- Engineered features from other multi-level categorical features and mixed features.

Make sure that for any new columns that you have engineered, that you've excluded the original columns from the final dataset. Otherwise, their values will interfere with the analysis later on the project. For example, you should not keep "PRAEGENDE_JUGENDJAHRE", since its values won't be useful for the algorithm: only the values derived from it in the engineered features you created should be retained. As a reminder, your data should only be from **the subset with few or no missing values**.

In [ ]:
# If there are other re-engineering tasks you need to perform, make sure you
# take care of them here. (Dealing with missing data will come in step 2.1.)



In [ ]:
# Do whatever you need to in order to ensure that the dataframe only contains
# the columns that should be passed to the algorithm functions.

# drop the original two columns that were used to create the engineered features as we no longer need them
good_df = good_df.drop(columns=["PRAEGENDE_JUGENDJAHRE", "CAMEO_INTL_2015"])

print(f"Cleaned shape: {good_df.shape}")
print(f"Columns: {good_df.shape[1]}")


### Step 1.3: Create a Cleaning Function

Even though you've finished cleaning up the general population demographics data, it's important to look ahead to the future and realize that you'll need to perform the same cleaning steps on the customer demographics data. In this substep, complete the function below to execute the main feature selection, encoding, and re-engineering steps you performed above. Then, when it comes to looking at the customer data in Step 3, you can just run this function on that DataFrame to get the trimmed dataset in a single step.

In [ ]:
def clean_data(df):
    """
    Perform feature trimming, re-encoding, and engineering for demographics
    data

    INPUT: Demographics DataFrame
    OUTPUT: Trimmed and cleaned demographics DataFrame

    Decisions (column drops, row threshold, multi-level drops) are fixed from
    the general-population analysis so the same cleaning is applied to any
    demographics dataset (e.g. customers).
    """

    df_clean = df.copy()

    # We need a list of outlier columns that we will drop, row threshold, 
    # and multi-level features that we will drop so we can apply the same cleaning 
    # to the customer dataset that we did to the general population dataset:
    outlier_columns = [
        "TITEL_KZ",
        "AGER_TYP",
        "KK_KUNDENTYP",
        "KBA05_BAUMAX",
        "GEBURTSJAHR",
        "ALTER_HH",
    ]
    row_missing_threshold = 25
    multi_level_features = [
        "CJT_GESAMTTYP",
        "FINANZTYP",
        "GFK_URLAUBERTYP",
        "LP_FAMILIE_FEIN",
        "LP_FAMILIE_GROB",
        "LP_STATUS_FEIN",
        "LP_STATUS_GROB",
        "NATIONALITAET_KZ",
        "SHOPPER_TYP",
        "ZABEOTYP",
        "GEBAEUDETYP",
        "CAMEO_DEUG_2015",
        "CAMEO_DEU_2015",
    ]

    # convert missing/unknown codes to NaN
    for _, row in feat_info.iterrows():
        feature = row["attribute"]
        if feature not in df_clean.columns:
            continue

        missing_raw = row["missing_or_unknown"]
        if not isinstance(missing_raw, str) or missing_raw == "[]":
            continue

        if re.search(r"[A-Za-z]", str(missing_raw)):
            missing_raw = re.sub(
                r"(?<=[\[,])\s*([A-Za-z]+)\s*(?=[,\]])",
                r"'\1'",
                str(missing_raw),
            )

        missing_codes = ast.literal_eval(missing_raw)
        if missing_codes:
            df_clean[feature] = df_clean[feature].replace(missing_codes, np.nan)

    # drop the same high-missing columns identified on the general population
    df_clean = df_clean.drop(
        columns=[c for c in outlier_columns if c in df_clean.columns]
    )

    # keep rows with few missing values (<=25 NaNs per row)
    missing_per_row = df_clean.isnull().sum(axis=1)
    good_rows = missing_per_row[missing_per_row <= row_missing_threshold]
    df_clean = df_clean.loc[good_rows.index].copy()

    # re-encode categoricals and drop the same multi-level columns as general pop
    if "OST_WEST_KZ" in df_clean.columns:
        df_clean["OST_WEST_KZ"] = df_clean["OST_WEST_KZ"].map({"O": 0, "W": 1})

    df_clean = df_clean.drop(
        columns=[c for c in multi_level_features if c in df_clean.columns]
    )

    # engineer mixed-type features
    decade_map = {
        1: 40, 2: 40,
        3: 50, 4: 50,
        5: 60, 6: 60, 7: 60,
        8: 70, 9: 70,
        10: 80, 11: 80, 12: 80, 13: 80,
        14: 90, 15: 90,
    }
    mainstream = {1, 3, 5, 8, 10, 12, 14}
    avantgarde = {2, 4, 6, 7, 9, 11, 13, 15}

    if "PRAEGENDE_JUGENDJAHRE" in df_clean.columns:
        col = "PRAEGENDE_JUGENDJAHRE"
        df_clean[f"{col}_DECADE"] = df_clean[col].map(decade_map)

        def youth_movement(code):
            if code in avantgarde:
                return 1
            if code in mainstream:
                return 0
            return np.nan

        df_clean[f"{col}_MOVEMENT"] = df_clean[col].map(youth_movement)

    if "CAMEO_INTL_2015" in df_clean.columns:
        col = "CAMEO_INTL_2015"
        wealth_map = {
            "11": 1, "12": 1, "13": 1, "14": 1, "15": 1,
            "21": 2, "22": 2, "23": 2, "24": 2, "25": 2,
            "31": 3, "32": 3, "33": 3, "34": 3, "35": 3,
            "41": 4, "42": 4, "43": 4, "44": 4, "45": 4,
            "51": 5, "52": 5, "53": 5, "54": 5, "55": 5,
            "XX": np.nan,
        }
        life_stage_map = {
            "11": 1, "12": 2, "13": 3, "14": 4, "15": 5,
            "21": 1, "22": 2, "23": 3, "24": 4, "25": 5,
            "31": 1, "32": 2, "33": 3, "34": 4, "35": 5,
            "41": 1, "42": 2, "43": 3, "44": 4, "45": 5,
            "51": 1, "52": 2, "53": 3, "54": 4, "55": 5,
            "XX": np.nan,
        }
        df_clean[f"{col}_WEALTH"] = df_clean[col].map(wealth_map)
        df_clean[f"{col}_LIFE_STAGE"] = df_clean[col].map(life_stage_map)

    # drop original mixed columns that were engineered
    engineered_originals = [
        col for col in ["PRAEGENDE_JUGENDJAHRE", "CAMEO_INTL_2015"] if col in df_clean.columns
    ]
    df_clean = df_clean.drop(columns=engineered_originals)

    return df_clean


In [ ]:
# test the new function and compare shape and columns to the manually cleaned dataframe
newly_cleaned_df = clean_data(gen_demo)

print("Testing the new function and comparing shape, columns, and values...\n")

if newly_cleaned_df is None:
    raise RuntimeError(
        "clean_data returned None. Re-run the clean_data cell above — "
        "the function must end with return df_clean."
    )

print(f"clean_data shape: {newly_cleaned_df.shape}")
print(f"good_df shape:    {good_df.shape}")

assert newly_cleaned_df.shape == good_df.shape, (
    f"Shape mismatch: {newly_cleaned_df.shape} vs {good_df.shape}"
)
assert set(newly_cleaned_df.columns) == set(good_df.columns), (
    "Column mismatch between clean_data() and manual good_df"
)

print("\nNew function's shape and columns matches what we created in the cells above ... woohoo!")


# now let's do a value-by-value comparison
assert newly_cleaned_df.equals(good_df), "Data mismatch between clean_data() and manual good_df"

print("\nValues also match ... woohoo!")



## Step 2: Feature Transformation

### Step 2.1: Apply Feature Scaling

Before we apply dimensionality reduction techniques to the data, we need to perform feature scaling so that the principal component vectors are not influenced by the natural differences in scale for features. Starting from this part of the project, you'll want to keep an eye on the [API reference page for sklearn](http://scikit-learn.org/stable/modules/classes.html) to help you navigate to all of the classes and functions that you'll need. In this substep, you'll need to check the following:

- sklearn requires that data not have missing values in order for its estimators to work properly. So, before applying the scaler to your data, make sure that you've cleaned the DataFrame of the remaining missing values. This can be as simple as just removing all data points with missing data, or applying an [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) to replace all missing values. You might also try a more complicated procedure where you temporarily remove missing values in order to compute the scaling parameters before re-introducing those missing values and applying imputation. Think about how much missing data you have and what possible effects each approach might have on your analysis, and justify your decision in the discussion section below.
- For the actual scaling function, a [StandardScaler](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) instance is suggested, scaling each feature to mean 0 and standard deviation 1.
- For these classes, you can make use of the `.fit_transform()` method to both fit a procedure to the data as well as apply the transformation to the data at the same time. Don't forget to keep the fit sklearn objects handy, since you'll be applying them to the customer demographics data towards the end of the project.

In [ ]:
# checking the missing values
print(f"Missing values in good_df: {good_df.isnull().sum().sum()}")
print(f"Rows with any missing: {good_df.isnull().any(axis=1).sum()}")


In [ ]:
# If you've not yet cleaned the dataset of all NaN values, then investigate and
# do that now.

# using the imputer to replace the missing values
imputer = SimpleImputer(strategy="median")
good_imputed = pd.DataFrame(
    imputer.fit_transform(good_df),
    columns=good_df.columns,
    index=good_df.index
)


In [ ]:
# checking the missing values again after imputation
print(f"Missing values in good_imputed: {good_imputed.isnull().sum().sum()}")
print(f"Rows with any missing: {good_imputed.isnull().any(axis=1).sum()}")

In [ ]:
# apply feature scaling to the general population demographics data.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
good_scaled = pd.DataFrame(
    scaler.fit_transform(good_imputed),
    columns=good_imputed.columns,
    index=good_imputed.index
)
print(f"Shape before scaling: {good_imputed.shape}")
print(f"Shape after scaling: {good_scaled.shape}\n")

print("Scaled data statistics:")
print(good_scaled.describe().loc[["mean", "std"]])


### Discussion 2.1: Apply Feature Scaling

Only ~22% of rows had NaNs, so I decided to use imputation rather than dropping rows. Median imputation was used to fill the gaps, which is robust to skew/outliers.

For scaling, the standard scaler was used. PCA is sensitive to feature scale and high variance columns would have dominated without the scaling.  



### Step 2.2: Perform Dimensionality Reduction

On your scaled data, you are now ready to apply dimensionality reduction techniques.

- Use sklearn's [PCA](http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) class to apply principal component analysis on the data, thus finding the vectors of maximal variance in the data. To start, you should not set any parameters (so all components are computed) or set a number of components that is at least half the number of features (so there's enough features to see the general trend in variability).
- Check out the ratio of variance explained by each principal component as well as the cumulative variance explained. Try plotting the cumulative or sequential values using matplotlib's [`plot()`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.plot.html) function. Based on what you find, select a value for the number of transformed features you'll retain for the clustering part of the project.
- Once you've made a choice for the number of components to keep, make sure you re-fit a PCA instance to perform the decided-on transformation.

In [ ]:
from sklearn.decomposition import PCA

print("Starting PCA")
# good_scaled is the df that has been scaled and imputed
print(f"The type of good_scaled is: {type(good_scaled)}")

# n_features is the number of features in the scaled df
n_features = good_scaled.shape[1]

# pca_full is the PCA instance 
pca_full = PCA(n_components=n_features, random_state=42)

# good_pca is the transformed data from the PCA instance
good_pca = pca_full.fit_transform(good_scaled)

print(f"Input shape: {good_scaled.shape}")
print(f"PCA output shape: {good_pca.shape}")
print(f"The type of the PCA output is: {type(good_pca)}")
print(f"Number of components: {pca_full.n_components_}")


In [ ]:
# Investigate the variance accounted for by each principal component.

# .explained_variance_ratio_ returns the variance explained by each principal component
variances = pca_full.explained_variance_ratio_

# np.cumsum returns the cumulative sum of the elements
cumulative_variances = np.cumsum(variances)

print(f"Variance explained by first 10 components: {variances[:10]}\n")
print(f"Cumulative variance at 10 components: {cumulative_variances[9]:.3f}")
print(f"Cumulative variance at 20 components: {cumulative_variances[19]:.3f}")
print(f"Cumulative variance at 30 components: {cumulative_variances[29]:.3f}")
print(f"Cumulative variance at 40 components: {cumulative_variances[39]:.3f}")
print(f"Cumulative variance at 50 components: {cumulative_variances[49]:.3f}")
print(f"Cumulative variance at 60 components: {cumulative_variances[59]:.3f}")


# bar plot of variance explained for each component
dimensions = [f"PC{i + 1}" for i in range(len(variances))]
# create a dataframe with the variance explained to plot the bar chart
variance_df = pd.DataFrame({"variance": variances}, index=dimensions)

# create a figure with two subplots (bar plot and line plot)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# bar plot of variance explained for each component
variance_df.plot(ax=axes[0], kind="bar", legend=False, color="orange")
axes[0].set_title("Variance explained for principal components")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Explained variance")
axes[0].set_xticklabels(dimensions, rotation=90)

# line plot: cumulative variance explained
axes[1].plot(range(1, len(cumulative_variances) + 1), cumulative_variances, marker="o", markersize=3, color="orange")

# add horizontal lines for various cumulative variances
axes[1].axhline(0.80, color="darkblue", label="80%", linestyle="--")
axes[1].axhline(0.85, color="darkgreen", label="85%", linestyle="--")
axes[1].axhline(0.90, color="blue", label="90%")
axes[1].axhline(0.95, color="green", label="95%")

# set the title, x-axis label, y-axis label, and legend
axes[1].set_title("Cumulative variance explained")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative explained variance ratio")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# First let's do some housekeeping otherwise the udacity workspace will run out of memory and crash

# we only need lightweight refs for later cells (so we can free the big frames)
azdias_columns = gen_demo.columns          # cell: compare customer columns
azdias_n_rows = len(gen_demo)              # cell: we need this for step 3.3 high-missing totals (as we delete gen_demo below)
feature_cols = good_df.columns             # cell: align/transform customers
feature_names = good_scaled.columns        # PC weight mapping / cluster interpretation

# now let's free up memory for the large objects we no longer need before the second PCA fit 
# otherwise the udacity workspace will crash when we run the PCA / fit_transform :(

del good_pca          # full 68-component transform 
del gen_demo          # original demographics 
del gen_demo_clean    # cleaned full demographics
del good_imputed      # imputed copy 
del good_df           # only columns needed later (feature_cols) will be kept
del pca_full          # variance plot already done above
# we still need *good_scaled* for the PCA fit_transform and will delete it after 

# delete other small intermediates 
for _name in [
    "bad_df", "good_rows_df", "bad_rows_df", "newly_cleaned_df",
    "missing_mask", "missing_corr", "missing_corr_sorted",
    "variance_df", "df",
]:
    if _name in globals():
        del globals()[_name]

gc.collect() # garbage collection this frees up memory


In [ ]:
# Re-apply PCA to the data while selecting for number of components to retain.

# based on the plot above and aiming for a min 90% cumulative variance I decided to retain 35 components
pca_90 = PCA(n_components=35, random_state=42)
# fit the PCA instance to the scaled data with 35 components
good_pca = pca_90.fit_transform(good_scaled)

print(f"Input shape: {good_scaled.shape}")
print(f"PCA output shape: {good_pca.shape}")
print(f"The type of the PCA output is: {type(good_pca)}")
print(f"Number of components: {pca_90.n_components_}")


In [ ]:
# scaled matrix good_scaled is no longer needed (only feature_names are kept) let's remove it
del good_scaled
gc.collect() # one more time to free up memory after we deleted the good_scaled 


### Discussion 2.2: Perform Dimensionality Reduction

PCA was applied to the scaled general-population data (good_scaled), which has 68 features and 797961 rows. Initially it was applied to all 68 components to examine the variance across components. The first components capture the majority of the variance (PC1 ~17%, PC2 ~13%), with a drop-off after those early components. So cumulative variance grows quickly at first and then slows down rapidly:

- 10 components: 61.8%
- 20 components: 77.8%
- 30 components: 87.3%
- 40 components: 93.1%
- 50 components: 96.8%
- 60 components: 99.2%

Based on these variance findings I chose 35 principal components that capture ~90% of the original variance. 

The dimensionality was therefore reduced from 68 to 35 features which keeps most information with nearly half the features. 

The fitted pca_90 object was saved for later use on customer data.


### Step 2.3: Interpret Principal Components

Now that we have our transformed principal components, it's a nice idea to check out the weight of each variable on the first few components to see if they can be interpreted in some fashion.

As a reminder, each principal component is a unit vector that points in the direction of highest variance (after accounting for the variance captured by earlier principal components). The further a weight is from zero, the more the principal component is in the direction of the corresponding feature. If two features have large weights of the same sign (both positive or both negative), then increases in one tend expect to be associated with increases in the other. To contrast, features with different signs can be expected to show a negative correlation: increases in one variable should result in a decrease in the other.

- To investigate the features, you should map each weight to their corresponding feature name, then sort the features according to weight. The most interesting features for each principal component, then, will be those at the beginning and end of the sorted list. Use the data dictionary document to help you understand these most prominent features, their relationships, and what a positive or negative value on the principal component might indicate.
- You should investigate and interpret feature associations from the first three principal components in this substep. To help facilitate this, you should write a function that you can call at any time to print the sorted list of feature weights, for the *i*-th principal component. This might come in handy in the next step of the project, when you interpret the tendencies of the discovered clusters.

In [ ]:
# Map weights for the first principal component to corresponding feature names
# and then print the linked values, sorted by weight.
# HINT: Try defining a function here or in a new cell that you can reuse in the
# other cells.

def map_pca_weights(pca, feature_names, component):
    """
    Return feature weights for one principal component. Sorted descending.
    """
    weights = pd.Series(pca.components_[component], index=feature_names) 
    # this creates a series of the weights for the component with the feature names as the index
    weights.sort_values(ascending=False, inplace=True) 
    # sort them descending
    return weights

# 0 indexed -> first principal component
pc1_weights = map_pca_weights(pca_90, feature_names, 0)
print(pc1_weights)


In [ ]:
# Map weights for the second principal component to corresponding feature names
# and then print the linked values, sorted by weight.

# 1 = second principal component
pc2_weights = map_pca_weights(pca_90, feature_names, 1)
print(pc2_weights)


In [ ]:
# Map weights for the third principal component to corresponding feature names
# and then print the linked values, sorted by weight.

# 2 = third principal component
pc3_weights = map_pca_weights(pca_90, feature_names, 2)
print(pc3_weights)


### Discussion 2.3: Interpret Principal Components

#### The first component PC1 (~17% variance)

##### Top 5 positive weighted features:

- MOBI_REGIO (Movement patterns)                 
- PLZ8_ANTG1 (Number of 1-2 family houses in the PLZ8 region)                     
- KBA05_ANTG1 (Number of 1-2 family houses in the microcell)                       
- KBA05_GBZ (Number of buildings in the microcell)                        
- FINANZ_MINIMALIST (low financial interest)  

##### 5 most negatively weighted features:

- PLZ8_ANTG3 (Number of 6-10 family houses in the PLZ8 region)
- PLZ8_ANTG4 (Number of 10+ family houses in the PLZ8 region)
- PLZ8_BAUMAX (Most common building type within the PLZ8 region with 1 being small houses, 5 being business buildings)
- CAMEO_INTL_2015_WEALTH (Wealth)
- ORTSGR_KLS9 (Size of community)

##### Intuition:

PC1 seems to separate lower-density, smaller-household areas (positive) from dense, multi-unit, lower-wealth urban areas (negative).


#### The second component PC2 (~13% variance)

##### Top 5 positive weighted features:

- ALTERSKATEGORIE_GROB (Estimated age based on given name analysis)
- FINANZ_VORSORGER (Financial typology: be prepared)
- SEMIO_ERL (Personality typology: event-oriented)
- SEMIO_LUST (Personality typology: sensual-minded)
- RETOURTYP_BK_S (Return type)

##### 5 most negatively weighted features:

- SEMIO_REL (Personality typology: religious)
- PRAEGENDE_JUGENDJAHRE_DECADE (Decade of dominating youth movement; engineered from PRAEGENDE_JUGENDJAHRE)
- FINANZ_SPARER (Financial typology: money-saver)
- SEMIO_TRADV (Personality typology: traditional-minded)
- SEMIO_PFLICHT (Personality typology: dutiful)

##### Intuition:

PC2 seems to separate mostly on life stage and financial preparedness. With older (positive) vs younger (negative), financially prepared (positive) vs saver oriented (negative) and religious/traditional/dutiful (all negative).

#### The third component PC3 (~9% variance)

##### Top 5 positive weighted features:

- ANREDE_KZ (Gender)
- SEMIO_KAEM (Personality typology: combative attitude)
- SEMIO_DOM (Personality typology: dominant-minded)
- SEMIO_KRIT (Personality typology: critical-minded)
- SEMIO_RAT (Personality typology: rational)

##### 5 most negatively weighted features:

- SEMIO_VERT (Personality typology: dreamful)
- SEMIO_SOZ (Personality typology: socially-minded)
- SEMIO_FAM (Personality typology: family-minded)
- SEMIO_KULT (Personality typology: cultural-minded)
- FINANZ_MINIMALIST (Financial typology: low financial interest)

##### Intuition:

PC3 seems to separate mostly on gender and personality type with combative/dominant/critical (positive) vs. dreamful/social/family/cultural (negative) personalities. 

#### Can we interpret positive and negative values from them in a meaningful way?

Examining the top 5 vs. bottom 5 features in PC1, PC2, and PC3 shows clearly that the signs are meaningful. A high (positive) score on a PC means a person/area looks more like the positively weighted features; a low (negative) score means more like the negatively weighted features. Across PC1–PC3 that maps cleanly to neighborhood/wealth, age/life-stage, and gender/personality opposites.

## Step 3: Clustering

### Step 3.1: Apply Clustering to General Population

You've assessed and cleaned the demographics data, then scaled and transformed them. Now, it's time to see how the data clusters in the principal components space. In this substep, you will apply k-means clustering to the dataset and use the average within-cluster distances from each point to their assigned cluster's centroid to decide on a number of clusters to keep.

- Use sklearn's [KMeans](http://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans) class to perform k-means clustering on the PCA-transformed data.
- Then, compute the average difference from each point to its assigned cluster's center. **Hint**: The KMeans object's `.score()` method might be useful here, but note that in sklearn, scores tend to be defined so that larger is better. Try applying it to a small, toy dataset, or use an internet search to help your understanding.
- Perform the above two steps for a number of different cluster counts. You can then see how the average distance decreases with an increasing number of clusters. However, each additional cluster provides a smaller net benefit. Use this fact to select a final number of clusters in which to group the data. **Warning**: because of the large size of the dataset, it can take a long time for the algorithm to resolve. The more clusters to fit, the longer the algorithm will take. You should test for cluster counts through at least 10 clusters to get the full picture, but you shouldn't need to test for a number of clusters above about 30.
- Once you've selected a final number of clusters to use, re-fit a KMeans instance to perform the clustering operation. Make sure that you also obtain the cluster assignments for the general demographics data, since you'll be using them in the final Step 3.3.

In [ ]:
# do a test run with 2 clusters to understand the score function
kmeans = KMeans(n_clusters=2, random_state=42, n_init="auto")
kmeans.fit(good_pca)
score = kmeans.score(good_pca)
print(f"Score for 2 clusters: {score}")

# do a test run with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
kmeans.fit(good_pca)
score = kmeans.score(good_pca)
print(f"Score for 3 clusters: {score}")

# also see https://stackoverflow.com/questions/32370543/understanding-score-returned-by-scikit-learn-kmeans
# to understand the score function ...


In [ ]:
# Over a number of different cluster counts...
from sklearn.cluster import KMeans

# Test a range of cluster counts (at least through 10; keep it modest since the dataset is large)
cluster_counts = list(range(1, 16))
scores = []

# loop over the range of cluster counts
for n_clusters in cluster_counts:
    # run k-means clustering on the data and...
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
    kmeans.fit(good_pca) # fit the model to the data

    # compute the average within-cluster distances.
    # .score() returns the "Sum of distances of samples to their closest cluster center."
    # (larger = better).
    score = kmeans.score(good_pca)
    scores.append(score)
    print(f"Clusters: {n_clusters} Score: {score}")


In [ ]:
# Investigate the change in within-cluster distance across number of clusters.
# HINT: Use matplotlib's plot function to visualize this relationship.

plt.figure(figsize=(10, 6))
plt.plot(cluster_counts, scores, color="orange")
plt.xlabel("Number of clusters")
plt.ylabel("Score (Sum of distances of samples to their closest cluster center)")
plt.title("KMeans score vs. number of clusters")
plt.xticks(cluster_counts)
plt.grid(True) # display a grid for better readability
plt.show()


In [ ]:
# Re-fit the k-means model with the selected number of clusters and obtain
# cluster predictions for the general population demographics data.

# Pick k from the elbow plot above (change this after inspecting the curve)
n_clusters_final = 8

kmeans_final = KMeans(n_clusters=n_clusters_final, random_state=42, n_init="auto")
general_preds = kmeans_final.fit_predict(good_pca)

cluster_counts = pd.Series(general_preds).value_counts().sort_index()
cluster_pcts = (cluster_counts / len(good_pca) * 100).round(2)

print(f"Fitted KMeans k={n_clusters_final}")
print(f"Cluster assignment counts:\n{cluster_counts}")
print(f"\nTotal: {cluster_counts.sum()} vs. our len(good_pca): {len(good_pca)}")
print(f"\nPercentages:\n{cluster_pcts}")
print(f"\nTotal percentage: {cluster_pcts.sum():.0f}%")


### Discussion 3.1: Apply Clustering to General Population

K-Means was run on the PCA-transformed general population data (`good_pca`, 797961 rows and 35 components) for cluster counts from 1 to 15. For each k the KMeans score was recorded. As expected the score improves as k increases, but the gains get smaller after the early values of k.

Looking at the elbow plot, there is a soft elbow around 8–12 clusters. After about k=8 the reductions in cluster distance slowly flatten. I therefore chose 8 clusters as a balance between data size and information captured.

I then re-fit KMeans with k=8 and obtained cluster assignments for the full general-population set. All 8 clusters are populated, with sizes ranging from ~8% to ~17% of the data (largest cluster 1, smallest 6).


### Step 3.2: Apply All Steps to the Customer Data

Now that you have clusters and cluster centers for the general population, it's time to see how the customer data maps on to those clusters. Take care to not confuse this for re-fitting all of the models to the customer data. Instead, you're going to use the fits from the general population to clean, transform, and cluster the customer data. In the last step of the project, you will interpret how the general population fits apply to the customer data.

- Don't forget when loading in the customers data, that it is semicolon (`;`) delimited.
- Apply the same feature wrangling, selection, and engineering steps to the customer demographics using the `clean_data()` function you created earlier. (You can assume that the customer demographics data has similar meaning behind missing data patterns as the general demographics data.)
- Use the sklearn objects from the general demographics data, and apply their transformations to the customers data. That is, you should not be using a `.fit()` or `.fit_transform()` method to re-fit the old objects, nor should you be creating new sklearn objects! Carry the data through the feature scaling, PCA, and clustering steps, obtaining cluster assignments for all of the data in the customer demographics data.

In [ ]:
# Load in the customer demographics data.
customers = pd.read_csv("data/Udacity_CUSTOMERS_Subset.csv", sep=";") # loading the udacity customer data
print(customers.shape)
customers.head()


In [ ]:
# compare the columns in our original gen_demo with the columns in the customers data
# (azdias_columns saved earlier so we could free gen_demo from memory)
missing_cols = set(azdias_columns) - set(customers.columns)
extra_cols = set(customers.columns) - set(azdias_columns)

print(f"Columns in gen_demo but not in customers: {missing_cols}")
print(f"Columns in customers but not in gen_demo: {extra_cols}")


In [ ]:
# Apply preprocessing, feature transformation, and clustering from the general
# demographics onto the customer data, obtaining cluster predictions for the
# customer demographics data.

# Use our clean_data function from above (same fixed decisions as general population)
customers_clean = clean_data(customers)
print(f"Customers after clean_data: {customers_clean.shape}")
print(f"Rows removed by clean_data: {len(customers) - len(customers_clean)}")

# Columns must already match the general-population feature set (no NaN reindex patch)
missing_cols = set(feature_cols) - set(customers_clean.columns)
extra_cols = set(customers_clean.columns) - set(feature_cols)
assert not missing_cols, f"Customers missing columns vs general pop: {sorted(missing_cols)}"
assert not extra_cols, f"Customers have unexpected extra columns: {sorted(extra_cols)}"

# Reorder only (same columns, same meaning as training)
customers_aligned = customers_clean[feature_cols]
print(f"Customers aligned: {customers_aligned.shape}")

# Transform with the *fitted* general-population objects (no refitting)
customers_imputed = pd.DataFrame(
    imputer.transform(customers_aligned),
    columns=feature_cols,
    index=customers_aligned.index,
)

print(f"Customers after imputation: {customers_imputed.shape}")

customers_scaled = pd.DataFrame(
    scaler.transform(customers_imputed),
    columns=feature_cols,
    index=customers_imputed.index,
)
customers_pca = pca_90.transform(customers_scaled)

print(f"Customers after PCA: {customers_pca.shape}")

# now call our fitted kmeans model from the general population on the PCA transformed customer data
customer_preds = kmeans_final.predict(customers_pca)

# now we can count the number of customers in each cluster
customer_counts = pd.Series(customer_preds).value_counts().sort_index()
# and calculate the percentage of customers in each cluster
customer_pcts = (customer_counts / len(customer_preds) * 100).round(2)

print(f"\nCustomers PCA shape: {customers_pca.shape}")
print(f"Customer cluster counts:\n{customer_counts}")
print(f"\nCustomer cluster percentages:\n{customer_pcts}")


### Step 3.3: Compare Customer Data to Demographics Data

At this point, you have clustered data based on demographics of the general population of Germany, and seen how the customer data for a mail-order sales company maps onto those demographic clusters. In this final substep, you will compare the two cluster distributions to see where the strongest customer base for the company is.

Consider the proportion of persons in each cluster for the general population, and the proportions for the customers. If we think the company's customer base to be universal, then the cluster assignment proportions should be fairly similar between the two. If there are only particular segments of the population that are interested in the company's products, then we should see a mismatch from one to the other. If there is a higher proportion of persons in a cluster for the customer data compared to the general population (e.g. 5% of persons are assigned to a cluster for the general population, but 15% of the customer data is closest to that cluster's centroid) then that suggests the people in that cluster to be a target audience for the company. On the other hand, the proportion of the data in a cluster being larger in the general population than the customer data (e.g. only 2% of customers closest to a population centroid that captures 6% of the data) suggests that group of persons to be outside of the target demographics.

Take a look at the following points in this step:

- Compute the proportion of data points in each cluster for the general population and the customer data. Visualizations will be useful here: both for the individual dataset proportions, but also to visualize the ratios in cluster representation between groups. Seaborn's [`countplot()`](https://seaborn.pydata.org/generated/seaborn.countplot.html) or [`barplot()`](https://seaborn.pydata.org/generated/seaborn.barplot.html) function could be handy.
  - Recall the analysis you performed in step 1.1.3 of the project, where you separated out certain data points from the dataset if they had more than a specified threshold of missing values. If you found that this group was qualitatively different from the main bulk of the data, you should treat this as an additional data cluster in this analysis. Make sure that you account for the number of data points in this subset, for both the general population and customer datasets, when making your computations!
- Which cluster or clusters are overrepresented in the customer dataset compared to the general population? Select at least one such cluster and infer what kind of people might be represented by that cluster. Use the principal component interpretations from step 2.3 or look at additional components to help you make this inference. Alternatively, you can use the `.inverse_transform()` method of the PCA and StandardScaler objects to transform centroids back to the original data space and interpret the retrieved values directly.
- Perform a similar investigation for the underrepresented clusters. Which cluster or clusters are underrepresented in the customer dataset compared to the general population, and what kinds of people are typified by these clusters?

In [ ]:
# Compare the proportion of data in each cluster for the customer data to the
# proportion of data in each cluster for the general population.



# Include high-missing rows as an extra cluster (-1). In Discussion 1.1.3 these
# rows looked qualitatively different, so let's treat them as their own
# segment.
HIGH_MISSING_CLUSTER = -1 # this is the cluster for the high-missing rows

# get the total number of rows in the general and customer datasets
# gen_demo was deleted earlier to free memory; use the saved row count
n_general_total = azdias_n_rows if "azdias_n_rows" in globals() else 891221
# get the total number of rows in the customer dataset
n_customer_total = len(customers)

# get the counts of the clusters in the general and customer datasets
general_counts = pd.Series(general_preds).value_counts()
# get the counts of the clusters in the customer dataset
customer_counts = pd.Series(customer_preds).value_counts()

# the new extra cluster (-1) that we created for the rows removed before because they had too many missing values
general_counts[HIGH_MISSING_CLUSTER] = n_general_total - len(general_preds)
customer_counts[HIGH_MISSING_CLUSTER] = n_customer_total - len(customer_preds)

# get the proportions of the clusters in the general and customer datasets
general_props = (general_counts / n_general_total).sort_index()
customer_props = (customer_counts / n_customer_total).sort_index()

print(
    f"High-missing (cluster {HIGH_MISSING_CLUSTER}): "
    f"general={general_counts[HIGH_MISSING_CLUSTER]} "
    f"({general_props[HIGH_MISSING_CLUSTER]*100:.2f}%), "
    f"customers={customer_counts[HIGH_MISSING_CLUSTER]} "
    f"({customer_props[HIGH_MISSING_CLUSTER]*100:.2f}%)"
)

# create a dataframe with the proportions of the general and customer data
comparison = pd.DataFrame({
    "general_percent": (general_props * 100).round(2),
    "customer_percent": (customer_props * 100).round(2),
})

# calculate the difference in proportions between the customer and general data
comparison["difference_prop"] = (comparison["customer_percent"] - comparison["general_percent"]).round(2)

# calculate the ratio of the customer and general data
comparison["ratio_cust_to_gen"] = (comparison["customer_percent"] / comparison["general_percent"]).round(2)

print("\nProportions of data in each cluster (incl. high-missing as -1):")
print(comparison)

# plot the proportions of the general and customer data
plot_df = (
    comparison[["general_percent", "customer_percent"]]
    .reset_index().rename(columns={"index": "cluster"})
    # .melt to rearrange the side by side bars (wide to long format)
    .melt(id_vars="cluster", var_name="datasets", value_name="percent")
)

plot_df["datasets"] = plot_df["datasets"].replace({
    "general_percent": "General population",
    "customer_percent": "Customers",
})

plt.figure(figsize=(12, 6))
sns.barplot(
    data=plot_df,
    x="cluster",
    y="percent",
    hue="datasets",  # split the bars by the two datasets
    palette=["orange", "gray"],
)
plt.xlabel("Cluster (-1 = high-missing rows)")
plt.ylabel("Percent of rows in cluster")
plt.title("Proportions: general population vs. customers (incl. high-missing)")
plt.show()


# plot the ratio of customers to general population
plt.figure(figsize=(12, 6))
sns.barplot(
    data=comparison.reset_index().rename(columns={"index": "cluster"}),
    x="cluster",
    y="ratio_cust_to_gen",
    color="orange",
)
plt.axhline(1.0, color="gray", linestyle="--")  # 1 = same share as general pop
plt.ylabel("Customer % / General %")
plt.xlabel("Cluster (-1 = high-missing rows)")
plt.title("Cluster representation (customers vs. general population)")
plt.show()

# Get the clusters that are overrepresented in the customers data compared to the general population
# using the comparison dataframe we created above.
# comparison[comparison["difference_prop"] > 0] basically filters the rows where the difference between
# the customer and general population is greater than 0
over = comparison[comparison["difference_prop"] > 0].sort_values("difference_prop", ascending=False)

# Get the clusters that are underrepresented in the customers data compared to the general population
# same as above but for the underrepresented clusters -> < 0
under = comparison[comparison["difference_prop"] < 0].sort_values("difference_prop")

# KMeans clusters only (exclude high-missing -1) for centroid / PC interpretation helpers
over_kmeans = over[over.index >= 0]
under_kmeans = under[under.index >= 0]

print("\nOverrepresented in customers:")
print(over)
print("\nUnderrepresented in customers:")
print(under)


In [ ]:
# helper functions for the next two cells


# Findings from "Discussion" in 2.3 as a lookup table so we can print the PC scores and their interpretations
PC_MEANINGS = {
    0: (
        "PC1 neighborhood / density / wealth: "
        "+ lower-density / smaller households; "
        "− dense multi-unit / lower-wealth urban"
    ),
    1: (
        "PC2 age / life stage / finances: "
        "+ older / financially prepared; "
        "− younger / saver / religious-traditional-dutiful"
    ),
    2: (
        "PC3 gender / personality: "
        "+ combative / dominant / critical; "
        "− dreamful / social-family-cultural"
    ),
}


def cluster_pc_scores(cluster_id, kmeans):
    """
    Return the cluster centroid's scores on the first n_pcs principal components.
    """
    n_pcs=3 # we want the first 3 PCs

    # get the centroid of the cluster
    center = kmeans.cluster_centers_[cluster_id]
    return pd.Series(
        center[:n_pcs], # center is the centroid of the cluster
        # scores for PC1, PC2, PC3 (when n_pcs=3)
        index=[f"PC{i + 1}" for i in range(n_pcs)],
    )


def show_cluster_pc_profile(cluster_ids, kmeans, pca, feature_names, n_features=5):
    """
    Interpret clusters via centroid PC scores with step 2.3 meanings (see PC_MEANINGS)
    """
    for cluster_id in cluster_ids:
        # high-missing segment has no KMeans centroid
        if cluster_id < 0:
            print("\n- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -")
            print(f"Interpreting Cluster {cluster_id} (high-missing rows)\n")
            if cluster_id in comparison.index:
                print("Cluster representation:")
                print(comparison.loc[cluster_id])
            print(
                "\nNo PCA/KMeans centroid — these rows were excluded in Step 1.1.3 "
                "because they had too many missing values and looked qualitatively different."
            )
            continue

        # get the centroid scores for the cluster
        scores = cluster_pc_scores(cluster_id, kmeans)
        print("\n- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -")
        
        print(f"Interpreting Cluster {cluster_id}\n")
        
        # comparison comes from the previous cell (cluster proportions)
        if cluster_id in comparison.index:
            print("Cluster representation:")
            print(comparison.loc[cluster_id])
        
        print("\nCentroid scores (PC1–PC3):")
        print(scores)

        # loop over the scores and print the PC1, PC2, PC3 scores and their interpretations (from Discussion 2.3)
        for i, score in enumerate(scores.to_numpy()):
            # get the direction of the score
            direction = "POSITIVE" if score >= 0 else "NEGATIVE"
            # print the PC1, PC2, PC3 scores and their interpretations
            print("\nInterpretation from Discussion 2.3:")
            print(f"- {PC_MEANINGS[i]}")
            print(f"\nCluster score: {score:.3f} ({direction})")
            
            # get the feature loadings for the PC
            weights = map_pca_weights(pca, feature_names, i)
            # if the score is positive, get the top 5 features
            if score >= 0:
                top = weights.head(n_features)
            else:
                top = weights.tail(n_features)
            print("\nFeatures:")
            print(top)


In [ ]:
# What kinds of people are part of a cluster that is overrepresented in the
# customer data compared to the general population?

# Use the cluster centroid's PC1–PC3 scores + interpretations we made in Discussion 2.3
# (cluster -1 = high-missing is summarized, not PCA-interpreted)
print("Overrepresented clusters:")

show_cluster_pc_profile(
    over.index, kmeans_final, pca_90, feature_names
)


In [ ]:
# What kinds of people are part of a cluster that is underrepresented in the
# customer data compared to the general population?

print("Underrepresented clusters:")

show_cluster_pc_profile(
    under.index, kmeans_final, pca_90, feature_names
)


### Discussion 3.3: Compare Customer Data to Demographics Data

The customer mix does not match the general population in terms of clustering / features. There are clear clusters that are clearly over- and under-represented.

Because high-missing rows looked qualitatively different in Discussion 1.1.3, they are treated here as an extra cluster (-1) for both datasets (not assigned via KMeans). Proportions use the full original row counts as denominators.

Cluster 1 is the most overrepresented KMeans segment with ~37% of customers (= popular with the mail-order company). The high-missing cluster (-1) is also strongly overrepresented with ~26%. On the other side clusters 5, 0, and 6 are almost absent with ~1.4%, ~0.8%, and ~0.7% (= unpopular with the mail-order company).

### The overrepresented clusters (popular)

#### Cluster 1

Cluster 1 has the strongest absolute overrepresentation (~15% general population vs. ~37% customers; factor ~2.5x)

- high PC1 which translates to lower-density / smaller households / less multi-unit urban
- weak positive PC2: slightly older / more financially prepared
- negative PC3: more dreamful / social / family / cultural

#### Cluster -1 (high-missing rows)

Cluster -1 is also strongly overrepresented (~10.5% general vs. ~26% customers; factor ~2.5x)

- high-missing rows (>25 missing values) kept as an extra segment, not assigned via KMeans
- qualitatively different in Discussion 1.1.3 (often one value dominates; possibly a default value)
- more common in the customer data than in the general population
- this is just a data-completeness finding (Step 3.3 requirement) and not a demographic target group like cluster 1

### The underrepresented clusters (unpopular)

#### Cluster 5

Almost absent (~11% general vs. ~1.4% customers)

- weak positive PC1: somewhat lower-density / smaller-household areas
- negative PC2: younger / saver / religious–traditional–dutiful
- positive PC3: combative / dominant / critical

#### Cluster 0

Almost absent (~9% general vs. ~0.8% customers)

- strong negative PC1: dense multi-unit / lower-wealth urban
- negative PC2: younger / saver / religious–traditional–dutiful
- positive PC3: combative / dominant / critical

#### Cluster 6

Almost absent (~7.5% general vs. ~0.7% customers)

- strong negative PC1: dense multi-unit / lower-wealth urban
- strong negative PC2: younger / saver / religious–traditional–dutiful
- negative PC3: dreamful / social / family / cultural

#### Cluster 3

Underrepresented (~8% general vs. ~2% customer)

- positive PC1: lower-density areas
- strong negative PC2: younger / saver / religious–traditional–dutiful
- negative PC3: dreamful / social / family / cultural

### Conclusion

Yes, you can clearly identify popular vs. unpopular segments.
The popular segments are cluster 1 and the high-missing cluster (-1), which together represent ~63% of the customers (vs. ~26% of the population).

The mail-order company is popular with:
- population that lives in lower-density or smaller-household areas (vs. dense multi-unit lower-wealth urban)
- slightly older population that is financially prepared, more dreamful, more social, more family-oriented
- also a larger share of high-missing / incomplete records than in the general population


> Congratulations on making it this far in the project! Before you finish, make sure to check through the entire notebook from top to bottom to make sure that your analysis follows a logical flow and all of your findings are documented in **Discussion** cells. Once you've checked over all of your work, you should export the notebook as an HTML document to submit for evaluation. You can do this from the menu, navigating to **File -> Download as -> HTML (.html)**. You will submit both that document and this notebook for your project submission.